# Bilişsel Performans Tahmini

Bu notebook'ta önce veriyi tanıyacağız, sonra sağlam bir model kurup tahmin yapacağız. Hedef: `bilissel_performans_skoru`.

Her adımda ne yaptığımızı ve neden yaptığımızı konuşarak ilerleyelim.

## 0. Kütüphaneler ve Kurulumlar

İhtiyacımız olan araçları yüklüyoruz. `catboost`, `lightgbm`, `optuna` ve `autogluon` ana modellerimiz.

In [ ]:
%pip install catboost lightgbm optuna autogluon.tabular -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping
import optuna
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')



## 1. Veriyi Yükleme ve İlk Bakış

Veriyi okuyup başını gösterelim, kaç satır, kaç sütun var kontrol edelim.

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test_x.csv')
test_ids = test_df['id'].copy()
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
train_df.head()

## 2. EDA – Keşifsel Veri Analizi

Veriyi anlamadan model kurmak olmaz. Bu bölümde eksik değerler, hedef değişkenin dağılımı, korelasyonlar ve kategorik değişkenlere bakacağız.

In [ ]:
# Eksik değer kontrolü
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Eksik değerler:")
    print(missing)
else:
    print("Hiç eksik değer yok.")

In [ ]:
# Hedef değişkenin dağılımı
plt.figure(figsize=(10,4))

# 1. Grafik: Histogram
plt.subplot(1,2,1)
train_df['bilissel_performans_skoru'].hist(bins=30, edgecolor='black')
plt.title('Hedef Değişken Histogramı')
plt.xlabel('Bilişsel Performans Skoru')
plt.ylabel('Frekans')

plt.subplot(1,2,2)
train_df['bilissel_performans_skoru'].plot.box() # .boxplot() yerine .plot.box() kullanıldı
plt.title('Kutu Grafiği - Hedef')

plt.tight_layout()
plt.show()

# İstatistiksel Özet
print(f"Ortalama: {train_df['bilissel_performans_skoru'].mean():.2f}")
print(f"Medyan: {train_df['bilissel_performans_skoru'].median():.2f}")
print(f"Standart Sapma: {train_df['bilissel_performans_skoru'].std():.2f}")
print(f"Min - Max: {train_df['bilissel_performans_skoru'].min()} - {train_df['bilissel_performans_skoru'].max()}")

In [ ]:
# Sayısal değişkenlerin korelasyon matrisi (hedef ile ilişkiler)
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if 'bilissel_performans_skoru' in numeric_cols:
    corr = train_df[numeric_cols].corr()['bilissel_performans_skoru'].sort_values(ascending=False)
    print("Hedef ile en yüksek korelasyon gösteren sayısal değişkenler:")
    print(corr.head(10))
    
    plt.figure(figsize=(10,8))
    sns.heatmap(train_df[numeric_cols].corr(), annot=False, cmap='coolwarm')
    
    plt.title('Sayısal Değişkenler Korelasyon Matrisi')
    plt.show()

In [ ]:
# Kategorik değişkenlerin frekansları ve hedef ile ilişkisi (boxplot)
cat_cols_raw = ["cinsiyet", "meslek", "ulke", "kronotip", "mevsim", "gun_tipi", "ruh_sagligi_durumu"]
for col in cat_cols_raw:
    if col in train_df.columns:
        print(f"\n{col} dağılımı:")
        print(train_df[col].value_counts())
        
        # Hedef değişkenin bu kategoriye göre kutu grafiği
        plt.figure(figsize=(8,4))
        train_df.boxplot(column='bilissel_performans_skoru', by=col)
        plt.title(f'{col} - Bilişsel Performans')
        plt.suptitle('')
        plt.xticks(rotation=45)
        plt.show()

In [ ]:
# Aykırı değerlere kısa bir bakış (IQR yöntemi ile)
for col in numeric_cols:
    if col != 'bilissel_performans_skoru' and col != 'id':
        Q1 = train_df[col].quantile(0.25)
        Q3 = train_df[col].quantile(0.75)
        IQR = Q3 - Q1
        outlier_count = ((train_df[col] < (Q1 - 1.5 * IQR)) | (train_df[col] > (Q3 + 1.5 * IQR))).sum()
        if outlier_count > 0:
            print(f"{col}: {outlier_count} aykırı değer")
            
            plt.figure(figsize=(6,3))
            train_df.boxplot(column=col)
            plt.title(f'{col} - Kutu Grafiği')
            plt.show()

EDA bitti. Şimdi gördüklerimiz ışığında feature engineering yapıp modellere hazırlanacağız.

## 3. Feature Engineering (Öznitelik Mühendisliği)

Ham verideki değişkenleri birbiriyle çarpıp bölerek, uyku kalitesi, stres etkileşimi gibi yeni anlamlı değişkenler türetiyoruz. Amaç, modelin ilişkileri daha kolay görmesini sağlamak. Kategorik değişkenleri de 'Bilinmiyor' ile dolduruyoruz ki testte görmediği bir değerle karşılaşınca takılmasın.

In [ ]:
def feature_engineering(df):
    d = df.copy()

    d['uyku_kalite']       = d['rem_yuzdesi'] + d['derin_uyku_yuzdesi']
    d['hafif_uyku']        = 100 - d['uyku_kalite']
    d['rem_derin_carpim']  = d['rem_yuzdesi'] * d['derin_uyku_yuzdesi']
    d['stres_calisma']     = d['stres_skoru'] * d['gunluk_calisma_saati']
    d['rem_stres_oran']    = d['rem_yuzdesi'] / (d['stres_skoru'] + 1)
    d['dinlenme_stres']    = d['uyku_kalite'] / (d['stres_skoru'] + 1)
    d['uyku_baskisi']      = d['gecelik_uyanma_sayisi'] * d['uykuya_dalma_suresi_dk']
    d['uyku_verimi']       = d['rem_derin_carpim'] / (d['gecelik_uyanma_sayisi'] + 1)
    d['kafein_ekran']      = (d['uyku_oncesi_kafein_mg'] / 100) + (d['uyku_oncesi_ekran_suresi_dk'] / 60)
    d['adim_calisma_oran'] = d['gunluk_adim_sayisi'] / (d['gunluk_calisma_saati'] + 1)

    stres_med  = d['stres_skoru'].median()
    rem_med    = d['rem_yuzdesi'].median()
    uyanma_med = d['gecelik_uyanma_sayisi'].median()

    d['lineer_skor'] = (
        (d['gun_tipi'] == 'Hafta sonu').astype(float) * 1.06 +
        d['stres_skoru'].fillna(stres_med) * (-0.59) +
        d['rem_yuzdesi'].fillna(rem_med) * 0.21 +
        d['gecelik_uyanma_sayisi'].fillna(uyanma_med) * (-0.18) +
        d['derin_uyku_yuzdesi'] * 0.10
    )

    yuksek = ['Yonetici', 'Lawyer']
    dusuk  = ['Satis ve Pazarlama Calisani']
    d['meslek_tier'] = d['meslek'].apply(
        lambda x: 2 if x in yuksek else (0 if x in dusuk else 1)
    )

    d['hafta_sonu']           = (d['gun_tipi'] == 'Hafta sonu').astype(int)
    d['stres_yuksek']         = (d['stres_skoru'] > 7).astype(int)
    d['stres_dusuk']          = (d['stres_skoru'] < 3).astype(int)
    d['ruh_sagligi_saglikli'] = (d['ruh_sagligi_durumu'] == 'Saglikli').astype(int)

    return d

cat_cols = ["cinsiyet", "meslek", "ulke", "kronotip", "mevsim", "gun_tipi", "ruh_sagligi_durumu"]

train_fe = feature_engineering(train_df)
test_fe  = feature_engineering(test_df)

for col in cat_cols:
    train_fe[col] = train_fe[col].fillna('Bilinmiyor').astype(str)
    test_fe[col]  = test_fe[col].fillna('Bilinmiyor').astype(str)

drop_cols = ['id', 'bilissel_performans_skoru', 'stres_grup', 'ruh_sagligi_encode']
X      = train_fe.drop(columns=drop_cols, errors='ignore')
y      = train_fe['bilissel_performans_skoru']
X_test = test_fe.drop(columns=[c for c in drop_cols if c != 'bilissel_performans_skoru'], errors='ignore')

print(f"Feature Engineering Sonunda | X: {X.shape} | X_test: {X_test.shape}")

## 4. Aykırı Değerlere Karşı Örnek Ağırlıkları

Aykırı değerleri silmek yerine, onlara düşük ağırlık veriyoruz. Önce basit bir CatBoost modeliyle 5 kat çapraz doğrulama yapıp tahmin hatalarını buluyoruz. En yüksek %5 hataya sahip örneklere 0.3, diğerlerine 1.0 ağırlık veriyoruz. Böylece model çok uç noktalara takılmadan genel performansı iyileştiriyor.

In [ ]:

kf_w  = KFold(n_splits=5, shuffle=True, random_state=42)
oof_w = np.zeros(len(X))

for tr_i, val_i in kf_w.split(X):
    m = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=6,
        verbose=0, cat_features=cat_cols, random_seed=42
    )
    m.fit(X.iloc[tr_i], y.iloc[tr_i])
    oof_w[val_i] = m.predict(X.iloc[val_i])

hatalar        = np.abs(y - oof_w)
esik           = np.percentile(hatalar, 95)
sample_weights = np.where(hatalar > esik, 0.3, 1.0)

print(f"Outlier eşiği: {esik:.4f}")
print(f"Düşük ağırlıklı satır: {(hatalar > esik).sum()} / {len(hatalar)}")

## 5. Optuna ile CatBoost Hiperparametre Optimizasyonu

CatBoost için en iyi hiperparametreleri bulmak üzere Optuna kullanıyoruz. 10 deneme (trial) yapıyoruz. Early stopping ile gereksiz yere uzamasını engelliyoruz. Bulduğumuz parametreleri daha sonra kullanacağız.

In [ ]:

kf_opt     = KFold(n_splits=5, shuffle=True, random_state=42)
tr_i, val_i = list(kf_opt.split(X))[0]
X_tr_o     = X.iloc[tr_i].copy()
X_vl_o     = X.iloc[val_i].copy()
y_tr_o     = y.iloc[tr_i]
y_vl_o     = y.iloc[val_i]
w_tr_o     = sample_weights[tr_i]

def objective(trial):
    p = {
        'iterations':            5000,
        'learning_rate':         trial.suggest_float('learning_rate', 0.01, 0.07, log=True),
        'depth':                 trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg':           trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature':   trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength':       trial.suggest_float('random_strength', 0, 2),
        'border_count':          trial.suggest_categorical('border_count', [128, 254]),
        'eval_metric':           'RMSE',
        'verbose':               0,
        'early_stopping_rounds': 200,
        'cat_features':          cat_cols,
        'random_seed':           42,
    }
    m = CatBoostRegressor(**p)
    m.fit(X_tr_o, y_tr_o,
          sample_weight=w_tr_o,
          eval_set=(X_vl_o, y_vl_o),
          use_best_model=True)
    return root_mean_squared_error(y_vl_o, m.predict(X_vl_o))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10, show_progress_bar=True)
best_p = study.best_params
print(f"Optuna tamamlandı | En iyi RMSE: {study.best_value:.5f}")
print(f"Parametreler: {best_p}")

## 6. CatBoost – İki Farklı Versiyon + Seed Averaging

Şimdi CatBoost'u üç farklı şekilde eğitiyoruz:
- Normal (hiperparametreler Optuna'dan)
- Sample Weight (önceden hesapladığımız ağırlıkları kullanarak)

Her versiyonu 5 farklı seed ile 5 kat çapraz doğrulama yaparak eğitiyoruz (seed averaging). Bu sayede rastgelelikten kaynaklanan varyansı düşürüyoruz. Sonra her versiyonun OOF performansına bakıp en iyisini seçiyoruz.

In [ ]:

SEEDS = [42, 43, 44, 45, 46]

base_params = {
    **best_p,
    'iterations':            6000,
    'eval_metric':           'RMSE',
    'verbose':               0,
    'early_stopping_rounds': 300,
    'cat_features':          cat_cols,
}

def run_catboost_seed_avg(params, use_sample_weight=False, label=""):
    all_test_preds = []
    all_rmse       = []
    oof_preds      = np.zeros(len(X))
    for s in SEEDS:
        kf      = KFold(n_splits=5, shuffle=True, random_state=s)
        s_preds = []
        s_rmse  = []
        for tr_i, val_i in kf.split(X):
            X_tr = X.iloc[tr_i].copy()
            X_vl = X.iloc[val_i].copy()
            y_tr = y.iloc[tr_i]
            y_vl = y.iloc[val_i]
            w_tr = sample_weights[tr_i] if use_sample_weight else None
            m = CatBoostRegressor(**{**params, 'random_seed': s})
            m.fit(X_tr, y_tr,
                  sample_weight=w_tr,
                  eval_set=(X_vl, y_vl),
                  use_best_model=True)
            val_pred = m.predict(X_vl)
            s_rmse.append(root_mean_squared_error(y_vl, val_pred))
            s_preds.append(m.predict(X_test))
            if s == 42:
                oof_preds[val_i] = val_pred
        all_test_preds.append(np.mean(s_preds, axis=0))
        all_rmse.append(np.mean(s_rmse))
        print(f"  [{label}] Seed {s} RMSE: {np.mean(s_rmse):.5f}")
    final_pred = np.mean(all_test_preds, axis=0)
    oof_score  = root_mean_squared_error(y, oof_preds)
    print(f"  [{label}] OOF RMSE: {oof_score:.5f}")
    return final_pred, oof_score

print("\n--- A) Normal ---")
cb_normal_pred, cb_normal_score = run_catboost_seed_avg(
    base_params, use_sample_weight=False, label="Normal"
)


print("\n--- C) Sample Weight ---")
cb_sw_pred, cb_sw_score = run_catboost_seed_avg(
    base_params, use_sample_weight=True, label="SampleW"
)

print("\nCatBoost Versiyon Karşılaştırması:")
print(f"  Normal:        {cb_normal_score:.5f}")
print(f"  Sample Weight: {cb_sw_score:.5f}")

cb_versions = {
    'normal': (cb_normal_score, cb_normal_pred),
    'sw':     (cb_sw_score,     cb_sw_pred),
}
best_cb_name  = min(cb_versions, key=lambda k: cb_versions[k][0])
cb_final      = cb_versions[best_cb_name][1]
cb_oof_score  = cb_versions[best_cb_name][0]
print(f"Kazanan CatBoost: {best_cb_name} ({cb_oof_score:.5f})")

## 7. LightGBM – Seed Averaging

LightGBM'i de benzer şekilde seed averaging ile eğitiyoruz. Kategorik değişkenleri `category` tipine çevirip, erken durdurma ile eğitiyoruz.

In [ ]:

X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

lgb_test_preds = []
lgb_oof_all    = np.zeros(len(X))
lgb_rmse_all   = []

for s in SEEDS:
    kf      = KFold(n_splits=5, shuffle=True, random_state=s)
    s_preds = []
    s_rmse  = []
    for tr_i, val_i in kf.split(X_lgb):
        X_tr = X_lgb.iloc[tr_i]
        X_vl = X_lgb.iloc[val_i]
        y_tr = y.iloc[tr_i]
        y_vl = y.iloc[val_i]
        m = LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.02,
            num_leaves=63,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=2.0,
            min_child_samples=20,
            random_state=s,
            verbosity=-1,
        )
        m.fit(X_tr, y_tr,
              eval_set=[(X_vl, y_vl)],
              callbacks=[lgb_early_stopping(200, verbose=False)])
        val_pred = m.predict(X_vl)
        s_rmse.append(root_mean_squared_error(y_vl, val_pred))
        s_preds.append(m.predict(X_test_lgb))
        if s == 42:
            lgb_oof_all[val_i] = val_pred
    lgb_test_preds.append(np.mean(s_preds, axis=0))
    lgb_rmse_all.append(np.mean(s_rmse))
    print(f"  Seed {s} RMSE: {np.mean(s_rmse):.5f}")

lgb_final     = np.mean(lgb_test_preds, axis=0)
lgb_oof_score = root_mean_squared_error(y, lgb_oof_all)
print(f"LightGBM OOF RMSE: {lgb_oof_score:.5f}")

## 8. AutoGluon

AutoGluon'u en kaliteli ayarlarla (best_quality) 2 saat boyunca eğitiyoruz. Kendi içinde bagging, stacking gibi birçok modeli deniyor ve harmanlıyor. Bu bizim en güçlü silahımız.

In [ ]:

from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label='bilissel_performans_skoru',
    eval_metric='rmse',
    verbosity=1
).fit(
    train_df.copy(),
    time_limit=7200,
    presets='best_quality',
    num_bag_folds=5,
    num_bag_sets=2,
    num_stack_levels=2,
)

ag_preds_raw = predictor.predict(test_df.copy())
ag_final     = np.clip(ag_preds_raw.values, 0, 10)
print(f"AutoGluon tamamlandı | [{ag_final.min():.3f}, {ag_final.max():.3f}]")

try:
    ag_oof_score = abs(predictor.leaderboard(silent=True).iloc[0]['score_val'])
    print(f"AG OOF (leaderboard): {ag_oof_score:.5f}")
except:
    ag_oof_score = 1.20564
    print(f"AG OOF (fallback):    {ag_oof_score:.5f}")

## 9. Blend Optimizasyonu

Üç modelimizin OOF performanslarına bakarak, en iyi ağırlık kombinasyonunu arıyoruz. Amaç: her bir modelin güçlü yanlarını birleştirip daha düşük RMSE elde etmek. Ağırlıkları grid search ile 0.05 adımlarla deniyoruz.

In [ ]:
print(f"  AG:  {ag_oof_score:.5f}")
print(f"  CB:  {cb_oof_score:.5f}")
print(f"  LGB: {lgb_oof_score:.5f}")

results = []
for ag_w in np.arange(0.30, 0.80, 0.05):
    for cb_w in np.arange(0.10, 0.60, 0.05):
        lgb_w = round(1.0 - ag_w - cb_w, 4)
        if lgb_w < 0.05 or lgb_w > 0.50:
            continue
        est = ag_w * ag_oof_score + cb_w * cb_oof_score + lgb_w * lgb_oof_score
        results.append((ag_w, cb_w, lgb_w, est))

results.sort(key=lambda x: x[3])
print("\nTop 5 blend:")
for ag_w, cb_w, lgb_w, est in results[:5]:
    print(f"    AG:{ag_w:.0%} CB:{cb_w:.0%} LGB:{lgb_w:.0%} -> {est:.5f}")

## 10. Submission Dosyalarının Oluşturulması

En iyi blend kombinasyonlarından 5 farklı submission dosyası üretiyoruz. Her tahmini 0-10 aralığına kırpıyoruz (performans skoru bu aralıkta tanımlı).

In [ ]:

cb_c  = np.clip(cb_final,  0, 10)
lgb_c = np.clip(lgb_final, 0, 10)

def save_sub(preds, fname):
    pd.DataFrame({
        'id': test_ids,
        'bilissel_performans_skoru': np.clip(preds, 0, 10)
    }).to_csv(fname, index=False)
    print(f"  {fname} kaydedildi")

# SUB 1: Mevcut en iyi referans
save_sub(ag_final * 0.60 + cb_c * 0.40,
         'sub_1_baseline_6040.csv')

# SUB 2: 3 model grid search #1
ag_w1, cb_w1, lgb_w1, _ = results[0]
save_sub(ag_final * ag_w1 + cb_c * cb_w1 + lgb_c * lgb_w1,
         f'sub_2_opt_{int(ag_w1*100)}ag_{int(cb_w1*100)}cb_{int(lgb_w1*100)}lgb.csv')

# SUB 3: 3 model grid search #2
ag_w2, cb_w2, lgb_w2, _ = results[1]
save_sub(ag_final * ag_w2 + cb_c * cb_w2 + lgb_c * lgb_w2,
         f'sub_3_opt_{int(ag_w2*100)}ag_{int(cb_w2*100)}cb_{int(lgb_w2*100)}lgb.csv')

# SUB 4: AG + LGB (CB'siz)
save_sub(ag_final * 0.65 + lgb_c * 0.35,
         'sub_4_ag65_lgb35.csv')

# SUB 5: Sadece en iyi CB
save_sub(cb_c, 'sub_5_catboost_only.csv')